# 🤖 Notebook: Retrieval-Augmented Generation (RAG) — *Part 2 of 3*

*This is the second of three notebooks in this chapter: 1) 2-Step RAG → 2) **Agentic RAG** → 3) Hybrid RAG. Start with `07_1_two_step_rag.ipynb` if you haven't yet — this notebook reuses its knowledge base and embedding cache.*

## 📚 Sources

- [LangChain Documentation: Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval)
- [LangChain Documentation: Build a Custom RAG Agent with LangGraph](https://docs.langchain.com/oss/python/langgraph/rag-agent)

## From 2-Step to Agentic RAG

At the end of the last notebook, we saw the core limitation of 2-Step RAG: it retrieves *every single time*, whether or not retrieval is actually useful for the question. Ask it "what's 12 times 7?" and it still goes and fetches three customer tweets nobody asked for.

**Agentic RAG** fixes this by not treating retrieval as a fixed pipeline step at all. Instead, retrieval becomes a **tool** — exactly like the tools you built in `05_intro_function_calling.ipynb` and `06_2_agents.ipynb` — and an LLM agent decides, as part of its own reasoning (the ReAct loop), *whether*, *when*, and *how many times* to call it. If the question needs no external knowledge, the agent just answers directly. If it needs information from multiple angles, it can call the retrieval tool several times with different queries.

The trade-off: we gain flexibility and efficiency, but lose the guarantee that retrieval happens at all, and we have no built-in way to check whether what *was* retrieved is actually good. (That's what `07_3_hybrid_rag.ipynb` adds back in.)

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

LLM_HOST = os.environ["LLM_HOST"]
LLM_URL = f"http://{LLM_HOST}:11434"
LLM_REASONING = "gemma4:26b"
EMBEDDING_MODEL = "embeddinggemma"

## Rebuilding the knowledge base

Same corpus as notebook 1: 500 real customer tweets about airlines, sampled once and saved to `content/airline_tweets.json`. We rebuild the `Document`s and the vector store here so this notebook can run standalone — but since we reuse the same `content/embedding_cache.json` file, none of the 500 embeddings need to be recomputed; they're all already cached from the last notebook.

In [2]:
import json
from langchain_core.documents import Document

with open("content/airline_tweets.json") as f:
    tweets = json.load(f)

docs = [Document(page_content=t["text"], metadata={"sentiment": t["sentiment"]}) for t in tweets]
print(f"Number of documents: {len(docs)}")

Number of documents: 500


In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

CACHE_PATH = "content/embedding_cache.json"


class CachedOllamaEmbeddings(OllamaEmbeddings):
    # See notebook 07_1 for a detailed explanation of how and why this works.
    cache_path: str = CACHE_PATH

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        cache = json.load(open(self.cache_path)) if os.path.exists(self.cache_path) else {}
        texts_to_embed = [t for t in texts if t not in cache]
        if texts_to_embed:
            print(f"Embedding {len(texts_to_embed)} new text(s) via the API...")
            for text, vector in zip(texts_to_embed, super().embed_documents(texts_to_embed)):
                cache[text] = vector
            json.dump(cache, open(self.cache_path, "w"))
        else:
            print("All texts already in cache, no API call needed.")
        return [cache[t] for t in texts]


embeddings = CachedOllamaEmbeddings(model=EMBEDDING_MODEL, base_url=LLM_URL)
vectorstore = InMemoryVectorStore(embeddings)
vectorstore.add_documents(docs)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

All texts already in cache, no API call needed.


## Wrapping retrieval as a tool

We use the same `@tool` decorator from `05_intro_function_calling.ipynb` / `06_2_agents.ipynb`. The docstring matters a lot here: it's the *only* information the agent has to decide whether this tool is relevant to a given question, so it needs to clearly describe what the tool searches over.

In [4]:
from langchain.tools import tool


@tool
def search_airline_tweets(query: str) -> str:
    """Search a database of real customer tweets about airlines for tweets relevant to the query.
    Use this whenever the question is about what customers are saying, complaining about, or praising."""
    results = retriever.invoke(query)
    return "\n\n".join(f"({d.metadata['sentiment']}) {d.page_content}" for d in results)

## Building the agent

`create_agent` from `06_2_agents.ipynb` — give it the LLM and the list of tools, and it handles the whole ReAct loop (decide → call tool(s) → observe → decide again → ... → final answer) for us.

In [5]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0)
agent = create_agent(model=llm, tools=[search_airline_tweets])

### Does the agent retrieve only when it should?

Let's compare a question that clearly needs the tweet database against one that clearly doesn't, and look at the full message trace to see whether the tool got called.

In [6]:
def run_and_trace(question):
    print(f"Q: {question}")
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    for m in result["messages"]:
        tool_calls = getattr(m, "tool_calls", None)
        if tool_calls:
            print(f"  {type(m).__name__}: calls tool {[tc['name'] + str(tc['args']) for tc in tool_calls]}")
        else:
            content = m.content if isinstance(m.content, str) else str(m.content)
            print(f"  {type(m).__name__}: {content[:200]!r}")
    print()


run_and_trace("What are customers saying about lost luggage?")
run_and_trace("What is 12 times 7?")

Q: What are customers saying about lost luggage?


Embedding 1 new text(s) via the API...


  HumanMessage: 'What are customers saying about lost luggage?'
  AIMessage: calls tool ["search_airline_tweets{'query': 'lost luggage'}"]
  ToolMessage: '(negative) @united still missing my luggage, was promised someone would call, no call so far, flight from Shanghai to DC with connecting flight in ORD\n\n(positive) @united Had wonderful, on time flight'
  AIMessage: 'Customers are expressing significant frustration regarding lost luggage, particularly with United Airlines. Based on recent social media activity, common complaints include:\n\n*   **Lack of Communicati'

Q: What is 12 times 7?


  HumanMessage: 'What is 12 times 7?'
  AIMessage: '12 times 7 is 84.'



For the first question, the agent calls `search_airline_tweets` before answering. For the second, it recognizes it doesn't need the tweet database at all and answers directly — no wasted retrieval call. This is exactly the decision 2-Step RAG *couldn't* make.

### Multi-step retrieval

Because retrieval is just a tool, the agent isn't limited to calling it once. If a question spans multiple topics, it can call the tool several times with different, more targeted queries — something a fixed 2-Step pipeline (one retrieval call per question) can't easily do either.

In [7]:
run_and_trace(
    "Compare what customers say about lost luggage with what they say about flight delays. "
    "Search for both topics separately."
)

Q: Compare what customers say about lost luggage with what they say about flight delays. Search for both topics separately.


All texts already in cache, no API call needed.


Embedding 1 new text(s) via the API...


  HumanMessage: 'Compare what customers say about lost luggage with what they say about flight delays. Search for both topics separately.'
  AIMessage: calls tool ["search_airline_tweets{'query': 'lost luggage'}"]
  ToolMessage: '(negative) @united still missing my luggage, was promised someone would call, no call so far, flight from Shanghai to DC with connecting flight in ORD\n\n(positive) @united Had wonderful, on time flight'
  AIMessage: calls tool ["search_airline_tweets{'query': 'flight delays'}"]
  ToolMessage: '(negative) @united lots of reports of system failures delaying flights over the last week. Currently sitting on the tarmac at OGG for over an hour.\n\n(negative) @AmericanAir right on cue with the delay'
  AIMessage: 'Based on recent customer tweets, here is a comparison of how passengers discuss lost luggage versus flight delays:\n\n### **Lost Luggage**\nCustomer sentiment regarding lost luggage is overwhelmingly **n'



## Exercise: Add a second tool

Give the agent a second tool, e.g. `count_tweets_by_sentiment(sentiment: str) -> int` that counts how many tweets in `docs` have a given sentiment label (`"positive"`, `"negative"`, or `"neutral"`) — no retrieval needed, just a lookup over `docs`' metadata. Then ask the agent something like *"How many negative tweets are there, and can you show me an example?"* and check the trace to see it use both tools appropriately.

In [8]:
# Insert code here...

<details>
<summary><b>Show solution</b></summary>

```python
@tool
def count_tweets_by_sentiment(sentiment: str) -> int:
    """Count how many tweets in the database have a given sentiment (positive, negative, or neutral)."""
    return sum(1 for d in docs if d.metadata["sentiment"] == sentiment)


agent_v2 = create_agent(model=llm, tools=[search_airline_tweets, count_tweets_by_sentiment])
run_and_trace("How many negative tweets are there, and can you show me an example?")
```

</details>

---

**Next up:** `07_3_hybrid_rag.ipynb` — the agent decides *whether* to retrieve, but never checks *how good* the retrieval was. Hybrid RAG adds that validation back in, as an explicit graph with grading and self-correction.